# SFT vs GRPO head-to-head: 500 HotpotQA validation questions

**Setup:** GPU preferred (T4, ~25 min total). CPU fallback works (~8 h).
Internet ON.

The 100-question ablation gave SFT-only=42.0, GRPO=44.0 EM. On 100
questions the standard error at p=0.43 is ~4.9 pp, so a 2-point gap
is well within noise. This notebook runs both adapters over the same
500 validation questions to get SE down to ~2.2 pp and see whether
the gap firms up or collapses.

Both adapters are evaluated with **identical** inference settings:
`max_steps=5, temperature=0.1`, same BM25 index, same question order.
Per-question correctness is saved for both adapters so a paired
(McNemar-style) test is possible after the fact.

## 1. Detect device

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = 'cuda'
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')
    print('Estimated runtime: ~25 min for 500 x 2 = 1000 rollouts')
else:
    DEVICE = 'cpu'
    print('No GPU - running on CPU.')
    print('Estimated runtime: ~8 h for 500 x 2 = 1000 rollouts')

print(f'DEVICE = {DEVICE}')

## 2. Clone repo & install

In [ ]:
import os

REPO_URL = 'https://github.com/202520030411/Research_Agent_RL.git'
REPO_DIR = '/kaggle/working/Research_Agent_RL'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
!pip install -q transformers peft accelerate datasets pyyaml tqdm bitsandbytes

## 3. Download both adapters

- **SFT adapter**: from `wuyue22/week3multigrpo` (this is the Week 1
  SFT run whose output Week 3 GRPO bootstrapped from).
- **GRPO adapter**: from `wuyue22/week3newgrpo` (the Week 3 final run,
  44.0% EM on the 100-q test).

In [ ]:
import os, shutil, glob

SFT_DIR  = 'checkpoints/qwen-sft/final'
GRPO_DIR = 'checkpoints/mt-grpo/final'

# --- SFT adapter ---
if not os.path.exists(SFT_DIR):
    print('Downloading SFT checkpoint from wuyue22/week3multigrpo...')
    !mkdir -p /tmp/sft_out
    !kaggle kernels output wuyue22/week3multigrpo -p /tmp/sft_out
    candidates = [
        '/tmp/sft_out/qwen-sft-adapter',
        '/tmp/sft_out/Research_Agent_RL/checkpoints/qwen-sft/final',
    ]
    src = next((c for c in candidates
                if os.path.exists(os.path.join(c, 'adapter_config.json'))), None)
    if src is None:
        ckpts = sorted(glob.glob('/tmp/sft_out/Research_Agent_RL/checkpoints/qwen-sft/checkpoint-*'))
        src = ckpts[-1] if ckpts else None
    if src is None:
        raise RuntimeError('No SFT checkpoint found')
    os.makedirs('checkpoints/qwen-sft', exist_ok=True)
    shutil.copytree(src, SFT_DIR)
    print(f'SFT adapter: {src} \u2192 {SFT_DIR}')
else:
    print(f'SFT adapter already present at {SFT_DIR}')

# --- GRPO adapter ---
if not os.path.exists(GRPO_DIR):
    print('Downloading GRPO checkpoint from wuyue22/week3newgrpo...')
    !mkdir -p /tmp/grpo_out
    !kaggle kernels output wuyue22/week3newgrpo -p /tmp/grpo_out
    candidates = [
        '/tmp/grpo_out/mt_grpo_adapter',
        '/tmp/grpo_out/Research_Agent_RL/checkpoints/mt-grpo/final',
    ]
    src = next((c for c in candidates
                if os.path.exists(os.path.join(c, 'adapter_config.json'))), None)
    if src is None:
        ckpts = sorted(glob.glob('/tmp/grpo_out/Research_Agent_RL/checkpoints/mt-grpo/epoch*'))
        src = ckpts[-1] if ckpts else None
    if src is None:
        raise RuntimeError('No GRPO checkpoint found')
    os.makedirs('checkpoints/mt-grpo', exist_ok=True)
    shutil.copytree(src, GRPO_DIR)
    print(f'GRPO adapter: {src} \u2192 {GRPO_DIR}')
else:
    print(f'GRPO adapter already present at {GRPO_DIR}')

print()
!ls -lh {SFT_DIR}
print()
!ls -lh {GRPO_DIR}

## 4. Load base model + tokenizer (once)

In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if DEVICE == 'cuda':
    load_kwargs = dict(device_map={'': 0}, torch_dtype=torch.bfloat16)
else:
    load_kwargs = dict(torch_dtype=torch.float32)

# Read base model name from either adapter (both target the same base)
with open(f'{SFT_DIR}/adapter_config.json') as f:
    base_model_name = json.load(f).get('base_model_name_or_path', 'Qwen/Qwen2.5-0.5B-Instruct')

tokenizer = AutoTokenizer.from_pretrained(SFT_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

print(f'Loading base: {base_model_name}  (dtype={load_kwargs["torch_dtype"]})')
base = AutoModelForCausalLM.from_pretrained(
    base_model_name, trust_remote_code=True, **load_kwargs,
)
if DEVICE == 'cpu':
    base = base.to('cpu')
print('Base model loaded.')

## 5. Load HotpotQA & build tool index (once)

In [ ]:
from datasets import load_dataset
from agent.tools import ToolExecutor

N_EVAL = 500

print('Loading HotpotQA...')
hf = load_dataset('hotpot_qa', 'distractor', trust_remote_code=True)
val_hf = hf['validation']

val_questions = [
    {'question': val_hf[i]['question'], 'answer': val_hf[i]['answer']}
    for i in range(N_EVAL)
]

tool_executor = ToolExecutor(top_k=2)
tool_executor.build_from_hotpotqa(val_hf, index_path='data/tool_index.jsonl')
print(f'Tool index: {len(tool_executor)} documents')
print(f'Val questions: {len(val_questions)}')

## 6. Eval helper

In [ ]:
from peft import PeftModel
from rl.multi_turn_grpo import collect_episode
from data.sft_dataset import SYSTEM_PROMPT
from tqdm import tqdm
import time, gc, torch

def run_eval(adapter_dir, adapter_name, base, tokenizer, val_questions, tool_executor, device):
    print(f'\n=== Evaluating {adapter_name} ===')
    t0 = time.time()
    model = PeftModel.from_pretrained(base, adapter_dir, is_trainable=False)
    model.eval()

    correct, total_steps = 0, 0
    per_q = []
    for i, q in enumerate(tqdm(val_questions, desc=adapter_name)):
        ep = collect_episode(
            q['question'], q['answer'],
            model, tokenizer, tool_executor,
            system_prompt=SYSTEM_PROMPT,
            max_steps=5, temperature=0.1, device=device,
        )
        correct     += int(ep.correct)
        total_steps += ep.n_steps
        per_q.append({
            'idx':      i,
            'question': ep.question,
            'gold':     ep.gold_answer,
            'correct':  ep.correct,
            'n_steps':  ep.n_steps,
        })

    elapsed = time.time() - t0
    n = len(val_questions)
    summary = {
        'adapter':       adapter_name,
        'adapter_dir':   adapter_dir,
        'max_steps':     5,
        'temperature':   0.1,
        'n_eval':        n,
        'accuracy':      correct / n,
        'avg_steps':     total_steps / n,
        'elapsed_s':     round(elapsed, 1),
    }
    print(f'  acc      : {summary["accuracy"]:.3f}')
    print(f'  avg_steps: {summary["avg_steps"]:.2f}')
    print(f'  elapsed  : {elapsed/60:.1f} min')

    # free memory before loading the next adapter
    del model
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()

    return summary, per_q

## 7. Run both evaluations back-to-back

In [ ]:
sft_summary,  sft_per_q  = run_eval(SFT_DIR,  'SFT-only',       base, tokenizer, val_questions, tool_executor, DEVICE)
grpo_summary, grpo_per_q = run_eval(GRPO_DIR, 'GRPO (Week 3)', base, tokenizer, val_questions, tool_executor, DEVICE)

## 8. Paired comparison & save

In [ ]:
import json, math

# Build paired contingency on per-question correctness
assert len(sft_per_q) == len(grpo_per_q) == N_EVAL
b10 = b01 = b11 = b00 = 0  # b10 = SFT right, GRPO wrong; etc.
for s, g in zip(sft_per_q, grpo_per_q):
    assert s['idx'] == g['idx']
    if s['correct']     and not g['correct']: b10 += 1
    elif not s['correct'] and g['correct']:   b01 += 1
    elif s['correct']     and g['correct']:   b11 += 1
    else:                                     b00 += 1

# McNemar (two-sided, normal approx.) on the discordant pairs
n_disc = b10 + b01
if n_disc > 0:
    z = (b01 - b10) / math.sqrt(n_disc)
    # two-sided p-value using simple erfc approximation
    p_two = math.erfc(abs(z) / math.sqrt(2))
else:
    z, p_two = 0.0, 1.0

# Per-adapter standard errors (Wald)
def se(p, n):
    return math.sqrt(p * (1 - p) / n)

print('=== 500-question comparison ===')
print(f'SFT-only   acc: {sft_summary["accuracy"]:.3f}  +/- {se(sft_summary["accuracy"],  N_EVAL):.3f} (1-SE)')
print(f'GRPO       acc: {grpo_summary["accuracy"]:.3f}  +/- {se(grpo_summary["accuracy"], N_EVAL):.3f} (1-SE)')
print(f'gap           : {grpo_summary["accuracy"] - sft_summary["accuracy"]:+.3f}')
print()
print(f'Paired contingency:')
print(f'  both right:              {b11}')
print(f'  both wrong:              {b00}')
print(f'  SFT right, GRPO wrong:   {b10}  (discordant)')
print(f'  GRPO right, SFT wrong:   {b01}  (discordant)')
print()
print(f'McNemar (two-sided normal approx.): z = {z:.3f}, p = {p_two:.4f}')

out = {
    'summary': {
        'sft':        sft_summary,
        'grpo':       grpo_summary,
        'gap':        grpo_summary['accuracy'] - sft_summary['accuracy'],
        'mcnemar_z':  z,
        'mcnemar_p':  p_two,
        'paired': {
            'both_right':                b11,
            'both_wrong':                b00,
            'sft_only_right':            b10,
            'grpo_only_right':           b01,
        },
    },
    'sft_per_q':  sft_per_q,
    'grpo_per_q': grpo_per_q,
}
with open('/kaggle/working/ablation_500_results.json', 'w') as f:
    json.dump(out, f, indent=2)
print('\nSaved \u2192 /kaggle/working/ablation_500_results.json')